# 01 · Extracción por API (BCRPData)
Genera `carpeta3/datos_crudos/datos_crudos_api_2024200482A.xlsx` (y .csv): **id | fecha | una columna por variable**.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Nombres y apellidos: ASTETE ROSALES GRAYCE NICOL
# Código de matrícula: 2024200482A
# Tema N.º 2 del temario: El margen de intermediación financiera en el Perú: spread TAMN-TIPMN y sus determinantes
# Fecha de extracción:2026-09-25
# ============================================================================
# 01_extraccion_api | Vía 1: API REST de BCRPData (Banco Central de Reserva del Perú)
# CELDA 1. Librerías y parámetros congelados (constantes, no fechas dinámicas)
# ============================================================================
import json, re, time, logging
from pathlib import Path
import pandas as pd
import requests

CODIGO_MATRICULA = "2024200482A"
FECHA_INICIO = "2021-01-01"
FECHA_CORTE  = "2025-12-31"          # ~1 240 días hábiles por variable
URL_BASE = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"
PAUSA = 1.0                          # segundos entre solicitudes
CABECERAS = {"User-Agent": f"UNCP-Finanzas-I-055D/1.0 (investigacion academica; e_{CODIGO_MATRICULA}@uncp.edu.pe)"}

# Determinantes del spread (todas DIARIAS en BCRPData)
# columna de la tabla : (código BCRP, descripción, unidad)
VARIABLES = {
    "tasa_referencia_bcrp_pct":  ("PD12301MD", "Tasa de referencia de la política monetaria del BCRP", "% anual"),
    "tasa_interbancaria_mn_pct": ("PD04692MD", "Tasa de interés interbancaria en moneda nacional (costo de fondeo)", "% anual"),
    "riesgo_pais_embig_pbs":     ("PD04709XD", "Spread EMBIG Perú (riesgo país)", "puntos básicos"),
    "tipo_cambio_venta_pen_usd": ("PD04638PD", "Tipo de cambio interbancario, venta", "S/ por US$"),
}

RAIZ = Path("/content/drive/MyDrive/Base de datos y código Grayce Nicol Astete Rosales")
DIR_JSON = RAIZ / "datos_crudos" / "json_bcrp"
DIR_JSON.mkdir(parents=True, exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S",
                    handlers=[logging.FileHandler(RAIZ / "log_ejecucion.txt", encoding="utf-8"),
                              logging.StreamHandler()], force=True)
log = logging.getLogger()

In [ ]:
# ============================================================================
# CELDA 2. Funciones
# ============================================================================
MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7,
         "ago": 8, "sep": 9, "set": 9, "oct": 10, "nov": 11, "dic": 12}

def fecha_bcrp(texto):
    """BCRPData entrega la fecha diaria como '04.Ene.21'. La convierte a fecha real."""
    m = re.match(r"(\d{1,2})\.([A-Za-z]{3})\.(\d{2,4})", str(texto).strip())
    if not m:
        return pd.NaT
    anio = int(m.group(3)); anio += 2000 if anio < 100 else 0
    return pd.Timestamp(anio, MESES[m.group(2).lower()], int(m.group(1)))

def valor_bcrp(texto):
    """Convierte el texto a número. 'n.d.' (no disponible) queda como celda vacía.
    El valor original exacto se conserva en el JSON guardado en json_bcrp/."""
    try:
        return float(texto)
    except (TypeError, ValueError):
        return None

def descargar_serie(codigo):
    """GET /api/{codigo}/json/{inicio}/{fin}/esp. Guarda el JSON exacto como evidencia."""
    url = f"{URL_BASE}/{codigo}/json/{FECHA_INICIO}/{FECHA_CORTE}/esp"
    for intento in range(1, 4):
        try:
            r = requests.get(url, headers=CABECERAS, timeout=90)
            log.info(f"GET {url} | HTTP {r.status_code}")
            r.raise_for_status()
            (DIR_JSON / f"{codigo}.json").write_bytes(r.content)
            datos = json.loads(r.content.decode("utf-8-sig"))
            return pd.Series({fecha_bcrp(p["name"]): valor_bcrp(p["values"][0]) for p in datos["periods"]})
        except json.JSONDecodeError:
            raise RuntimeError(f"{codigo}: la respuesta no es JSON (¿código inválido?)")
        except requests.exceptions.RequestException as e:
            log.info(f"{codigo}: error {e} (intento {intento})"); time.sleep(3 * intento)
    raise RuntimeError(f"No se pudo descargar {codigo}")

In [ ]:
# ============================================================================
# CELDA 3. Descarga y TABLA DE DATOS CRUDOS: id | fecha | una columna por variable
# ============================================================================
columnas = {}
for nombre, (codigo, descripcion, unidad) in VARIABLES.items():
    columnas[nombre] = descargar_serie(codigo)
    log.info(f"{codigo} -> {nombre}: {columnas[nombre].size} observaciones")
    time.sleep(PAUSA)

tabla_api = pd.DataFrame(columnas).sort_index().astype(float)
tabla_api.index.name = "fecha"
tabla_api = tabla_api.reset_index()
tabla_api["fecha"] = tabla_api["fecha"].dt.strftime("%Y-%m-%d")
tabla_api.insert(0, "id", range(1, len(tabla_api) + 1))

archivo = RAIZ / "datos_crudos" / f"datos_crudos_api_{CODIGO_MATRICULA}"
tabla_api.to_csv(f"{archivo}.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")  # se abre bien en Excel en español
tabla_api.to_excel(f"{archivo}.xlsx", index=False)
log.info(f"Tabla guardada: {archivo}.csv / .xlsx ({len(tabla_api)} filas)")

# Observaciones con dato por variable
conteo = pd.DataFrame({"observaciones": [tabla_api[c].notna().sum() for c in VARIABLES]},
                      index=list(VARIABLES))
conteo["cumple_1000"] = conteo["observaciones"] >= 1000
display(conteo)
display(tabla_api.head(10))

2026-09-25 01:04:47 | GET https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD12301MD/json/2021-01-01/2025-12-31/esp | HTTP 200
2026-09-25 01:04:48 | PD12301MD -> tasa_referencia_bcrp_pct: 1304 observaciones
2026-09-25 01:04:49 | GET https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD04692MD/json/2021-01-01/2025-12-31/esp | HTTP 200
2026-09-25 01:04:49 | PD04692MD -> tasa_interbancaria_mn_pct: 1304 observaciones
2026-09-25 01:04:51 | GET https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD04709XD/json/2021-01-01/2025-12-31/esp | HTTP 200
2026-09-25 01:04:51 | PD04709XD -> riesgo_pais_embig_pbs: 1304 observaciones
2026-09-25 01:04:53 | GET https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PD04638PD/json/2021-01-01/2025-12-31/esp | HTTP 200
2026-09-25 01:04:53 | PD04638PD -> tipo_cambio_venta_pen_usd: 1304 observaciones
2026-09-25 01:04:55 | Tabla guardada: /content/drive/MyDrive/Base de datos y código Grayce Nicol Astete Rosales/datos_crudos/datos_crudos_api

,observaciones,cumple_1000
tasa_referencia_bcrp_pct,1247,True
tasa_interbancaria_mn_pct,1245,True
riesgo_pais_embig_pbs,1304,True
tipo_cambio_venta_pen_usd,1246,True


,id,fecha,tasa_referencia_bcrp_pct,tasa_interbancaria_mn_pct,riesgo_pais_embig_pbs,tipo_cambio_venta_pen_usd
0,1,2021-01-01,NaN,NaN,132.0,NaN
1,2,2021-01-04,0.25,0.25,131.0,3.626667
2,3,2021-01-05,0.25,0.25,130.0,3.633667
3,4,2021-01-06,0.25,0.25,128.0,3.626833
4,5,2021-01-07,0.25,0.25,128.0,3.623000
5,6,2021-01-08,0.25,0.25,125.0,3.613833
6,7,2021-01-11,0.25,0.25,129.0,3.617333
7,8,2021-01-12,0.25,0.25,138.0,3.609000
8,9,2021-01-13,0.25,0.25,135.0,3.614000
9,10,2021-01-14,0.25,0.25,131.0,3.613500
